In [3]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

df = pd.DataFrame({
    'age': [25, None, 40, 35, None],
    'income': [30000, 45000, None, 80000, 52000],
    'city': ['Chennai', 'Mumbai', None, 'Chennai', 'Delhi'],
    'size': ['S', 'L', 'M', 'S', 'L'],
})

print('Initial DataFrame:')
print(df)

Initial DataFrame:
    age   income     city size
0  25.0  30000.0  Chennai    S
1   NaN  45000.0   Mumbai    L
2  40.0      NaN     None    M
3  35.0  80000.0  Chennai    S
4   NaN  52000.0    Delhi    L


In [4]:
age_imputer = SimpleImputer(strategy='median')
city_imputer = SimpleImputer(strategy='most_frequent')

df['age'] = age_imputer.fit_transform(df[['age']]).ravel()
df['city'] = city_imputer.fit_transform(df[['city']]).ravel()

print('\nAfter SimpleImputer on age (median) and city (most_frequent):')
print(df)


After SimpleImputer on age (median) and city (most_frequent):
    age   income     city size
0  25.0  30000.0  Chennai    S
1  35.0  45000.0   Mumbai    L
2  40.0      NaN     None    M
3  35.0  80000.0  Chennai    S
4  35.0  52000.0    Delhi    L


In [5]:
df_knn = df.copy()
df_knn['age'] = [25, np.nan, 40, 35, np.nan]
df_knn['income'] = [30000, 45000, np.nan, 80000, 52000]

knn_imputer = KNNImputer(n_neighbors=2)
df_knn[['age', 'income']] = knn_imputer.fit_transform(df_knn[['age', 'income']])

print('\nDataFrame with KNNImputer on numeric columns:')
print(df_knn)

print('\nComparison (SimpleImputer median vs KNNImputer):')
print(
    pd.DataFrame({
        'SimpleImputer_age': df['age'],
        'KNNImputer_age': df_knn['age'],
        'SimpleImputer_income': df['income'],
        'KNNImputer_income': df_knn['income'],
    })
)


DataFrame with KNNImputer on numeric columns:
    age   income     city size
0  25.0  30000.0  Chennai    S
1  30.0  45000.0   Mumbai    L
2  40.0  55000.0     None    M
3  35.0  80000.0  Chennai    S
4  30.0  52000.0    Delhi    L

Comparison (SimpleImputer median vs KNNImputer):
   SimpleImputer_age  KNNImputer_age  SimpleImputer_income  KNNImputer_income
0               25.0            25.0               30000.0            30000.0
1               35.0            30.0               45000.0            45000.0
2               40.0            40.0                   NaN            55000.0
3               35.0            35.0               80000.0            80000.0
4               35.0            30.0               52000.0            52000.0


In [6]:
ohe = OneHotEncoder(sparse_output=False)
city_encoded = ohe.fit_transform(df[['city']])
city_cols = ohe.get_feature_names_out(['city'])
df_ohe = pd.DataFrame(city_encoded, columns=city_cols, index=df.index)

df = pd.concat([df.drop(columns=['city']), df_ohe], axis=1)

print('\nAfter OneHotEncoder on city:')
print(df)

ordinal_enc = OrdinalEncoder(categories=[['S', 'M', 'L']])
df['size'] = ordinal_enc.fit_transform(df[['size']]).ravel()

print('\nAfter OrdinalEncoder on size (S=0, M=1, L=2):')
print(df)


After OneHotEncoder on city:
    age   income size  city_Chennai  city_Delhi  city_Mumbai  city_None
0  25.0  30000.0    S           1.0         0.0          0.0        0.0
1  35.0  45000.0    L           0.0         0.0          1.0        0.0
2  40.0      NaN    M           0.0         0.0          0.0        1.0
3  35.0  80000.0    S           1.0         0.0          0.0        0.0
4  35.0  52000.0    L           0.0         1.0          0.0        0.0

After OrdinalEncoder on size (S=0, M=1, L=2):
    age   income  size  city_Chennai  city_Delhi  city_Mumbai  city_None
0  25.0  30000.0   0.0           1.0         0.0          0.0        0.0
1  35.0  45000.0   2.0           0.0         0.0          1.0        0.0
2  40.0      NaN   1.0           0.0         0.0          0.0        1.0
3  35.0  80000.0   0.0           1.0         0.0          0.0        0.0
4  35.0  52000.0   2.0           0.0         1.0          0.0        0.0


In [7]:
df['target'] = ['No', 'Yes', 'Yes', 'No', 'Yes']

le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df['target'])

print('\nFinal Encoded DataFrame:')
print(df)


Final Encoded DataFrame:
    age   income  size  city_Chennai  city_Delhi  city_Mumbai  city_None  \
0  25.0  30000.0   0.0           1.0         0.0          0.0        0.0   
1  35.0  45000.0   2.0           0.0         0.0          1.0        0.0   
2  40.0      NaN   1.0           0.0         0.0          0.0        1.0   
3  35.0  80000.0   0.0           1.0         0.0          0.0        0.0   
4  35.0  52000.0   2.0           0.0         1.0          0.0        0.0   

  target  target_encoded  
0     No               0  
1    Yes               1  
2    Yes               1  
3     No               0  
4    Yes               1  
